In [ ]:
import os
import boto3
from sagemaker import get_execution_role
from pprint import pprint
import json
import time
import pandas as pd

### Constants

In [ ]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
# name of step function
str_name = 'example-state-machine-boto3'

### Write ```definition.json```

In [ ]:
%%writefile definition.json

{
  "Comment": "A description of my state machine",
  "StartAt": "Single Job",
  "States": {
    "Single Job": {
      "Type": "Task",
      "Resource": "arn:aws:states:::batch:submitJob.sync",
      "Arguments": {
        "JobName": "job-name-batch-example-single-1",
        "JobDefinition": "arn:aws:batch:us-west-2:ACCOUNT:job-definition/job-def-batch-example-single-1:1",
        "JobQueue": "arn:aws:batch:us-west-2:ACCOUNT:job-queue/queue-batch-example-single-1"
      },
      "Next": "Parallel Job"
    },
    "Parallel Job": {
      "Type": "Task",
      "Resource": "arn:aws:states:::batch:submitJob.sync",
      "Arguments": {
        "JobName": "job-name-batch-example-parallel-1",
        "JobDefinition": "arn:aws:batch:us-west-2:ACCOUNT:job-definition/job-def-batch-example-parallel-1:1",
        "JobQueue": "arn:aws:batch:us-west-2:ACCOUNT:job-queue/queue-batch-example-parallel-1",
        "ArrayProperties": {
          "Size": 10
        }
      },
      "Next": "Concatenate"
    },
    "Concatenate": {
      "Type": "Task",
      "Resource": "arn:aws:states:::lambda:invoke",
      "Output": "{% $states.result.Payload %}",
      "Arguments": {
        "FunctionName": "lambda-example-concat",
        "Payload": "{% $states.input %}"
      },
      "Retry": [
        {
          "ErrorEquals": [
            "Lambda.ServiceException",
            "Lambda.AWSLambdaException",
            "Lambda.SdkClientException",
            "Lambda.TooManyRequestsException"
          ],
          "IntervalSeconds": 1,
          "MaxAttempts": 3,
          "BackoffRate": 2,
          "JitterStrategy": "FULL"
        }
      ],
      "End": true
    }
  },
  "QueryLanguage": "JSONata"
}

### Make string definition

In [ ]:
# load it
dict_definition = json.load(open('./definition.json'))
# make into string
str_definition = json.dumps(dict_definition)

### Create state machine

In [ ]:
cls_client_sfn = boto3.client('stepfunctions')

In [ ]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

In [ ]:
# list state machines
dict_response = cls_client_sfn.list_state_machines(
)
list_dict_state_machines = dict_response['stateMachines']
list_dict_state_names = [{dict_state_machine['name']: dict_state_machine['stateMachineArn']} for dict_state_machine in list_dict_state_machines]
dict_state_names = {key: val for dict_name in list_dict_state_names for key, val in dict_name.items()}
pprint(dict_state_names)

In [ ]:
# get list of just names
list_str_names = [list(dict_state_names.keys())[0] for dict_state_names in list_dict_state_names]
# if our name is in there
if str_name in list_str_names:
    print(f'State machine {str_name} exists, it will be deleted')
    str_arn = dict_state_names[str_name]
    print(f'Deleting {str_arn}')
    print('')
    dict_response = cls_client_sfn.delete_state_machine(
        stateMachineArn=str_arn,
    )
    pprint(dict_response)
else:
    print(f'State machine {str_name} does not exist, so it will not be deleted')

In [ ]:
# make a state machine
while True:
    try:
        dict_response = cls_client_sfn.create_state_machine(
            name=str_name,
            definition=str_definition,
            roleArn=str_role,
            type='STANDARD',
        )
        pprint(dict_response)
        break
    except:
        pass

### Describe state machine

In [ ]:
str_state_machine_arn = dict_response['stateMachineArn']
print(f'State Machine ARN: {str_state_machine_arn}')
dict_response = cls_client_sfn.describe_state_machine(
    stateMachineArn=str_state_machine_arn,
)
pprint(dict_response)

### Execute step function workflow

In [ ]:
# # start execution
# dict_response = cls_client_sfn.start_execution(
#     stateMachineArn=str_state_machine_arn,
# )

### Clean-up

In [ ]:
os.remove('./definition.json')